Authenticate to BigQuery from Colab, import dependencies, create a BigQuery client for our GCP project, and define the fixed train/test split boundary (split_t) for the 1-day experiment.



```
# This is formatted as code
```

Notes: split_t is computed from the 1-day aggregated jobmetrics table as a 70/30 time split using start_time.

In [ ]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np

# 1. Authenticate the user
auth.authenticate_user()

# 2. Initialize the BigQuery client
project_id = 'traceadvisor-295a'
client = bigquery.Client(project=project_id)

#Computed a train/test split boundary
split_t = 233579300000

Query the aggregated 1-day table (borg_traces_1d_jobmetrics) and load only the required columns into a pandas dataframe.
We only keep rows with non-null request and peak usage fields so baseline evaluation is well-defined.
Output: df is our experiment dataset (one row per instance execution).

In [ ]:
query = f"""
SELECT
  recurring_job_id,
  collection_id,
  instance_index,
  start_time,
  end_time,
  req_cpu,
  req_mem,
  peak_cpu,
  peak_mem
FROM `traceadvisor-295a.traceadvisor_eda.borg_traces_1d_jobmetrics`
WHERE start_time IS NOT NULL
  AND req_cpu IS NOT NULL AND req_mem IS NOT NULL
  AND peak_cpu IS NOT NULL AND peak_mem IS NOT NULL
"""
df = client.query(query).to_dataframe()
print(df.shape)
df.head()

(2990353, 9)


,recurring_job_id,collection_id,instance_index,start_time,end_time,req_cpu,req_mem,peak_cpu,peak_mem
0,10163831006,10163831006,416,173100000000,259500000000,0.0,0.004448,0.143799,0.001488
1,10163831006,10163831006,475,173100000000,259500000000,0.0,0.004448,0.035278,0.001392
2,10163831006,10163831006,299,173100000000,259500000000,0.0,0.004448,0.080078,0.001553
3,10163831006,10163831006,603,173100000000,259500000000,0.0,0.004448,0.058167,0.001532
4,10163831006,10163831006,481,173100000000,259500000000,0.0,0.004448,0.039795,0.001511


Create execution_id for traceability and split the dataset into train and test based on start_time and the precomputed split_t.

time-aware split ensures that percentile estimates use only past executions and avoids data leakage.

In [ ]:
df["execution_id"] = df["collection_id"].astype(str) + "_" + df["instance_index"].astype(str)

train = df[df["start_time"] < split_t].copy()
test  = df[df["start_time"] >= split_t].copy()

print("Train:", train.shape, "Test:", test.shape)

Train: (2244698, 10) Test: (745655, 10)


Define a reusable evaluator that computes decision-quality metrics:
- violation rates (vr_cpu, vr_mem, vr_any)
- waste/slack under recommendation (waste_cpu, waste_mem)

Notes:
- A "violation" means observed peak usage exceeds recommended resources.
Waste is defined as max(0, recommendation − peak).

In [ ]:
def eval_method(df_eval, rec_cpu, rec_mem):
    rec_cpu = pd.Series(rec_cpu, index=df_eval.index)
    rec_mem = pd.Series(rec_mem, index=df_eval.index)

    viol_cpu = (df_eval["peak_cpu"] > rec_cpu).astype(int)
    viol_mem = (df_eval["peak_mem"] > rec_mem).astype(int)
    viol_any = ((viol_cpu == 1) | (viol_mem == 1)).astype(int)

    waste_cpu = np.maximum(0, rec_cpu - df_eval["peak_cpu"])
    waste_mem = np.maximum(0, rec_mem - df_eval["peak_mem"])

    return {
        "n_exec": int(len(df_eval)),
        "vr_cpu": float(viol_cpu.mean()),
        "vr_mem": float(viol_mem.mean()),
        "vr_any": float(viol_any.mean()),
        "waste_cpu": float(waste_cpu.sum()),
        "waste_mem": float(waste_mem.sum()),
    }

Evaluate Baseline 0, where the recommendation equals the original user request (rec = req).

Interpretation: this represents current practice and acts as a reliability baseline (often high violation risk if users under-request).

In [ ]:
b0 = eval_method(test, test["req_cpu"], test["req_mem"])
b0

{'n_exec': 745655,
 'vr_cpu': 0.9252120618784827,
 'vr_mem': 0.15309895326927334,
 'vr_any': 0.9271647075390094,
 'waste_cpu': 567.5624418258667,
 'waste_mem': 2671.6122512817383}

Compute per-recurring-job quantiles from TRAIN history only:
- p95_cpu, p99_cpu, p95_mem, p99_mem
- n_hist = number of historical runs available for that job.
- This is the core of Baseline 1 (“percentile sizing”).

In [ ]:
# Grouped quantiles from TRAIN only
q = train.groupby("recurring_job_id").agg(
    p95_cpu=("peak_cpu", lambda x: np.quantile(x, 0.95)),
    p99_cpu=("peak_cpu", lambda x: np.quantile(x, 0.99)),
    p95_mem=("peak_mem", lambda x: np.quantile(x, 0.95)),
    p99_mem=("peak_mem", lambda x: np.quantile(x, 0.99)),
    n_hist=("peak_cpu", "size")
).reset_index()

print(q.shape)
q.head()

(21558, 6)


,recurring_job_id,p95_cpu,p99_cpu,p95_mem,p99_mem,n_hist
0,++1ovjAez3LAfUHHeWmuJKZSg8cPni3cb42sErQPkAs=,0.009531,0.009951,0.000890,0.000905,17
1,++2esx+jWzddRnGUi5GIM/+H7j+vDgaj2Hl2gt5T9AA=,0.015845,0.017207,0.000664,0.000665,9
2,++kPZiCFqIsLiFb+2pjQ9EHfS5KL6Jn+UTHQJzvM7zY=,0.020721,0.020721,0.002926,0.002926,1
3,+/O+fbh5jgk1iuspu4QcIRRzU37WNFq8nS84SllItq4=,0.021545,0.021545,0.000808,0.000808,1
4,+1bp/nI0ZxDNvmTAJEoenRU4bIKm3gPCLkmXyY34N7w=,0.027191,0.027191,0.001616,0.001616,1


Create P95 recommendations for each TEST execution:
- If job has history: use p95_*
- If no history: fallback to original request (req_*)
- Then evaluate decision metrics using the evaluator.

In [ ]:
test_q = test.merge(q, on="recurring_job_id", how="left")

rec_cpu_p95 = test_q["p95_cpu"].fillna(test_q["req_cpu"])
rec_mem_p95 = test_q["p95_mem"].fillna(test_q["req_mem"])

b1_p95 = eval_method(test_q, rec_cpu_p95, rec_mem_p95)
b1_p95

{'n_exec': 745655,
 'vr_cpu': 0.25949668412335464,
 'vr_mem': 0.13170702268475368,
 'vr_any': 0.30691941983893356,
 'waste_cpu': 30437.161117362954,
 'waste_mem': 1398.334702920914}

Repeat the same Baseline 1 evaluation using P99 recommendations.

Comparison of P95 vs P99: shows the risk–efficiency tradeoff (P99 should reduce violations further but may increase waste).

In [ ]:
rec_cpu_p99 = test_q["p99_cpu"].fillna(test_q["req_cpu"])
rec_mem_p99 = test_q["p99_mem"].fillna(test_q["req_mem"])

b1_p99 = eval_method(test_q, rec_cpu_p99, rec_mem_p99)
b1_p99

{'n_exec': 745655,
 'vr_cpu': 0.23246541631183323,
 'vr_mem': 0.09955140111713863,
 'vr_any': 0.25522258953537497,
 'waste_cpu': 56064.93120368959,
 'waste_mem': 1867.9900374031072}

Build a 3-row summary table for:
- Baseline 0 (as-is)
- Baseline 1 P95
- Baseline 1 P99

Slack reduction is computed relative to Baseline 0 waste:
- (waste_baseline0 − waste_method) / waste_baseline0

In [ ]:
summary = pd.DataFrame([
    {"method": "baseline0_as_is", **b0},
    {"method": "baseline1_p95", **b1_p95},
    {"method": "baseline1_p99", **b1_p99},
])

base_cpu = summary.loc[summary.method=="baseline0_as_is","waste_cpu"].iloc[0]
base_mem = summary.loc[summary.method=="baseline0_as_is","waste_mem"].iloc[0]

summary["slack_reduction_cpu_pct"] = 100*(base_cpu - summary["waste_cpu"])/(base_cpu + 1e-9)
summary["slack_reduction_mem_pct"] = 100*(base_mem - summary["waste_mem"])/(base_mem + 1e-9)

summary

,method,n_exec,vr_cpu,vr_mem,vr_any,waste_cpu,waste_mem,slack_reduction_cpu_pct,slack_reduction_mem_pct
0,baseline0_as_is,745655,0.925212,0.153099,0.927165,567.562442,2671.612251,0.000000,0.000000
1,baseline1_p95,745655,0.259497,0.131707,0.306919,30437.161117,1398.334703,-5262.786343,47.659519
2,baseline1_p99,745655,0.232465,0.099551,0.255223,56064.931204,1867.990037,-9778.196137,30.080047


Assign confidence tiers using n_hist:
- high: ≥10 runs
- medium: 3–9 runs
- low: 0–2 runs

TraceAdvisor is designed to be more aggressive when evidence is strong and conservative when evidence is weak.

Output: tiered breakdown helps interpret results and safety.

In [ ]:
def tier(n):
    if pd.isna(n): return "low"
    n = int(n)
    if n >= 10: return "high"
    if n >= 3: return "medium"
    return "low"

test_q["tier"] = test_q["n_hist"].apply(tier)

# Example: baseline1_p95 metrics by tier
rows = []
for tname, g in test_q.groupby("tier"):
    rec_cpu = g["p95_cpu"].fillna(g["req_cpu"])
    rec_mem = g["p95_mem"].fillna(g["req_mem"])
    out = eval_method(g, rec_cpu, rec_mem)
    rows.append({"method":"baseline1_p95", "tier":tname, **out})
pd.DataFrame(rows).sort_values("tier")

,method,tier,n_exec,vr_cpu,vr_mem,vr_any,waste_cpu,waste_mem
0,baseline1_p95,high,553270,0.048085,0.072975,0.108186,30140.474255,649.225777
1,baseline1_p95,low,185633,0.893494,0.306082,0.900750,227.892655,746.932222
2,baseline1_p95,medium,6752,0.152399,0.150178,0.265255,68.794207,2.176704


Save summary results to a CSV file so they can be committed to the repo (reports/eval/) and used in the workbook/report.

Note: We save only summary outputs, not raw datasets.

In [ ]:
summary.to_csv("/content/summary_1d_experiments.csv", index=False)
print("Saved: /content/summary_1d_experiments.csv")

Saved: /content/summary_1d_experiments.csv
